# Does a protein's *chemistry* explain how confident AlphaFold is?

### An AP Biology / AP Chemistry investigation using real structural data

In 2021, an AI system called **AlphaFold** did something biologists had been trying to do
for fifty years: it learned to predict a protein's three-dimensional shape from nothing but
its amino acid sequence. Predictions for essentially every known protein are now free to
download.

Here's the part that makes it interesting for us. AlphaFold doesn't just hand you a shape —
it also tells you **how sure it is about every single amino acid**, on a scale from 0 to 100.
That score is called **pLDDT**. Some residues get a 98. Some get a 33. In the same protein.

So: **why?** What makes AlphaFold confident about one amino acid and unsure about the one
twelve positions later?

This notebook tests one specific idea:

> **Hypothesis.** A residue's *chemistry* — whether its side chain is nonpolar, polar,
> acidic, or basic — is related to how confidently AlphaFold can place it.

We'll find out that the answer is **yes, but not for the reason you'd first guess** — and
chasing down that "but" is where the real science happens.

No prior statistics required. Every statistical idea — standard deviation, confidence
interval, *p*-value, correlation — is explained from scratch as we need it.

---

## Where we're going

| Section | Question we're asking | New statistical tool |
|---|---|---|
| 1 | What does AlphaFold actually give us? | — |
| 2 | What are the 20 amino acids, chemically? | — |
| 3 | What is our protein made of? | Describing a distribution |
| 4 | Does confidence differ by side-chain class? | Confidence intervals, *t*-test, effect size |
| 5 | Can we compare all five classes at once? | One-way ANOVA |
| 6 | What does this look like in 3D? | — |
| 7 | Are nonpolar residues really buried? | Chi-square test |
| 8 | How strong is the relationship? | Regression, *r* and *r²* |
| 9 | Wait — is something else causing this? | Confounding variables |
| 10 | What did we actually learn? | Limitations |

Our main subject is **human lysozyme**, the enzyme in your tears and saliva that kills
bacteria by cutting open their cell walls. It's small (148 amino acids), extremely
well-studied, and completely folded — which makes it a clean place to start.

---

## Setup

We need five tools:

| Library | What we use it for |
|---|---|
| `requests` | downloading real data from the AlphaFold database over the internet |
| `numpy` | fast math on arrays of numbers |
| `pandas` | spreadsheet-like tables (`DataFrame`s) |
| `matplotlib` | every chart in this notebook |
| `scipy.stats` | the statistical tests |

### One important habit: cache your data

This notebook downloads data from live scientific databases. Those databases get updated,
go down for maintenance, and occasionally change their formats. If we re-downloaded
everything on every run, our numbers could silently change from one day to the next — and
in science, **a result you can't reproduce isn't really a result.**

So the `fetch_cached()` function below saves every download to a `data/` folder the first
time, then reads from that folder afterward. Two details worth noticing:

- `resp.raise_for_status()` makes the code **crash loudly** if a download fails. That sounds
  bad, but the alternative is worse: silently continuing with missing data and reporting
  confident conclusions built on nothing.
- The cached files get committed alongside this notebook, so anyone who downloads this
  repository gets the exact numbers you see here.

In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from scipy import stats

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)


def fetch_cached(url, filename, as_json=True):
    # Download `url` once, cache it in data/<filename>, and reuse it on later runs.
    # Raises on any failed download rather than quietly returning empty data.
    path = DATA_DIR / filename
    if path.exists():
        text = path.read_text(encoding="utf-8")
        source = "cache"
    else:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()          # fail loudly, never silently
        text = resp.text
        path.write_text(text, encoding="utf-8")
        source = "downloaded"
    print(f"  {filename:32s} ({source}, {len(text)/1000:.0f} KB)")
    return json.loads(text) if as_json else text


print("Setup complete.")

Setup complete.
